# 12장. Self-hosted 환경과 사내 패키지 설치

CSV 보고서 작업을 사내 패키지가 설치된 Docker executor에서 수행합니다.

| 연결 기준 | 이번 장 |
| --- | --- |
| 공통 예제 | 동일한 sales.csv의 매출 분석 |
| 새로 배울 기능 | Self-hosted 연결·명령과 패키지 실행 증거 |
| 이번 실습 조합 | Self-hosted Docker + 사내 패키지 + corporate-csv Skill |


<!-- colab-link -->
[독립 실행본 다운로드 안내](../../docs/colab.md) · 읽기용 · 실습은 로컬 Docker


## 학습 안내

이 장의 목표는 API가 만든 Self-hosted 환경을 내가 운영하는 Docker executor에 연결하고, 그 환경에서 사내 패키지와 Skill 코드가 실행됐는지 확인하는 것입니다. CSV → summary.json·report.md 예제를 사용합니다.

```bash
uv sync --locked
# Live Self-hosted API exercise: application key, executor key, and Docker required.
uv run --group dev python scripts/run_deployment.py --chapter 12 --live --docker
# Separate local package/Skill checks without API calls.
uv run --group dev python scripts/run_deployment.py --chapter 12 --docker --build
```

기본 API 경로는 이미지 준비 후 세션 생성 → Docker 준비 → 연결 확인 → Skill 실행 요청 → 결과 확보 → 정리의 6단계입니다. 이미지가 처음 빌드될 때는 시간이 걸릴 수 있습니다. 잘못된 해시 거절과 API 없는 로컬 Skill 실행은 본문 뒤의 선택 실습으로 분리합니다.

관찰 연결 복구는 [14장](14_operations.ipynb)에서 다룹니다. 이 장은 executor의 실제 연결·명령·파일을 확인하며, Skill 자동 선택 정책·실제 사내 VPN/인증/TLS·청구 대사를 검증하는 예제가 아닙니다.

[실습 안내와 검증 범위](../../docs/validation.md)

### 12–14장 실행 환경 연결

| 장 | 기본 API 실행 | 별도 선택 실습 |
| --- | --- | --- |
| 12장 | Self-hosted Docker executor | 로컬 패키지·Skill 점검 |
| 13장 | OpenAI-hosted + 애플리케이션 메모리 OTel | Collector·로컬 Hook 프로토콜 재생 |
| 14장 | OpenAI-hosted 관찰자 재연결 | 별도 Docker 프로세스 중단·파일 복원 |

13·14장은 이 장의 executor를 재사용하지 않고 새 hosted Session을 만듭니다. 공통 CSV를 유지하면서 관측·복구 기능에 집중합니다.


**실습 안내:** 각 장은 새 커널에서 독립 실행합니다. 실행 모드·응답과 파일의 차이·검증 기록은 [공통 학습·실습 안내](../../docs/validation.md)를 참고하세요.


## 0. 실행 모드

로컬에서 uv sync --locked 후 .venv 커널을 선택합니다. 세 실행 스위치는 기본 False이며 아래 모드 표에서 선택합니다. Colab은 읽기용이고 Docker 실습은 로컬에서 수행합니다.

선택 로컬 실습만 할 때는 API 준비·6단계·증거 표시 셀을 건너뛰고 이미지 준비와 선택 실습을 실행합니다. 두 경로를 켜면 API 정리 후 별도 worker에서 로컬 실습을 합니다.

CLI --docker만 지정하면 사본의 RUN_PACKAGE_CHECKS를 켜고 API 셀을 건너뜁니다. --live --docker는 기본 API 경로만 실행합니다. 오프라인 검증은 API와 Docker 모두 제외합니다.


In [ ]:
import os
import sys

if "google.colab" in sys.modules and os.environ.get("TUTORIAL_OFFLINE_CHECK") != "1":
    raise RuntimeError("This is the local source notebook. Upload the standalone version from the download guide to Colab.")
print("Use the local .venv kernel. Install dependencies with: uv sync --locked")


### API 코드와 Docker 준비 코드의 역할

API 세션 생성·이벤트·입력 제출·Turn/명령 조회는 노트북에서 직접 호출합니다. helper는 이미지 빌드·컨테이너/키 파일 준비·안전한 종료·파일 복사 같은 반복 작업을 담당합니다. Docker 준비와 executor 시작은 다른 단계입니다.


In [ ]:
import hashlib
import json
from contextlib import ExitStack
from tutorial_helpers.common import load_settings
from tutorial_helpers.deployment import (
    DockerLab,
    build_images,
    docker,
    SKILL_COMMAND,
    EXECUTOR_IMAGE,
    executor_key,
    prepare_executor,
    inspect_executor,
)


### 실행 모드와 입력 바이트

API 실습은 연결된 Docker에서 요청을 실행하는 경로이며 선택 로컬 실습은 패키지/Skill만 직접 실행하는 경로입니다. 아래 표로 모드를 선택하고 CSV는 원본 바이트 그대로 고정합니다.

| 경로 | RUN_LIVE | RUN_DOCKER | RUN_PACKAGE_CHECKS |
| --- | --- | --- | --- |
| API 기본 실습 | True | True | False |
| 선택 로컬 실습만 | False | True | True |
| 두 경로 | True | True | True |

로컬 실습의 성공은 API 연결 성공을 뜻하지 않습니다. 오프라인 검증에서는 API·Docker를 모두 실행하지 않습니다.


In [ ]:
RUN_DOCKER = False
RUN_LIVE = False
RUN_PACKAGE_CHECKS = False
MODEL_OVERRIDE = ""
CHAPTER = "12_self_hosted"
client = None
settings = load_settings(RUN_LIVE, MODEL_OVERRIDE)
root = settings.root
offline_check = os.environ.get("TUTORIAL_OFFLINE_CHECK") == "1"
docker_enabled = RUN_DOCKER and (not offline_check)
csv_bytes = (root / "data/sales.csv").read_bytes()
csv_text = csv_bytes.decode("utf-8")
image_input_sha256 = None
print({"api_enabled": settings.enabled, "docker_enabled": docker_enabled,
       "local_checks": RUN_PACKAGE_CHECKS, "input_sha256": hashlib.sha256(csv_bytes).hexdigest()})


In [ ]:
from tutorial_helpers.common import (
    new_session_metadata, cleanup_resources, cleanup_on_error, check_event, safe_error,
)
from uuid import uuid4
import time


In [ ]:
BASE_INSTRUCTIONS = (
    "You are a careful sales analyst. Use Python "
    "for all calculations. Use only the provided "
    "CSV; never invent rows. Do not browse the web"
    " or delegate. Keep progress commentary short."
    " Return only the JSON object requested by the"
    " user as the final answer, with no Markdown "
    "fences."
)
METRICS_CONTRACT = (
    'Return exactly {"row_count": integer, '
    '"total_quantity": integer, "total_revenue": '
    'integer, "by_product": {product_name: '
    "integer_revenue}}. Revenue is quantity "
    "multiplied by unit_price. Do not round "
    "integer amounts."
)


## 1. 실행 주체와 필수 준비

| 구성 | 이 예제에서 수행하는 일 |
| --- | --- |
| 노트북의 API 코드 | Session 생성·환경 연결 관찰·입력 제출·Turn/명령 이력 조회 |
| Docker 이미지 | Python·Codex·CSV·번들 Skill 파일 준비 |
| install_and_exec.py | 사내 패키지 설치 성공 후 codex exec-server 시작 |
| capability_directories | 컨테이너에 이미 있는 Skill 디렉터리 지정; ZIP 업로드가 아님 |
| executor | 연결된 환경에서 API가 요청한 명령 실행 |

앱 키는 노트북의 OPENAI_API_KEY, 전용 executor 키는 OPENAI_EXECUTOR_API_KEY입니다. 두 키는 분리하며 executor 키만 임시 파일로 읽기 전용 mount합니다. CODEX_HOME=/workspace/.codex는 컨테이너 내부 설정·상태 경로이며 Mac Codex 설치 경로가 아닙니다.

패키지 서버는 내부 Docker network에 있고 executor는 이 망과 bridge에 연결됩니다. 사내 패키지 접근 경로와 OpenAI 연결 경로는 별개입니다. HTTP/trusted-host는 가상 로컬 인덱스의 설정이며 실제 회사 인증·TLS 설정은 추가로 필요합니다.

작업 지시문은 번들 스크립트 실행을 명시합니다. 패키지 설치는 API의 자동 설치가 아니라 컨테이너 시작 코드에서 수행합니다.


### 파일·패키지가 실행 환경에 들어가는 경로

| 프로젝트 파일 | 실행 환경에서의 역할 |
| --- | --- |
| `data/sales.csv` | 빌드 시 `/workspace/sales.csv`로 복사 |
| `deploy/skills/corporate-csv/` | `/opt/capabilities/skills/corporate-csv/`에 배치 |
| `deploy/private_package/` | 실습용 acme-sales wheel 원본 |
| `deploy/prepare.py` | wheel·패키지 인덱스·해시 목록 생성 |
| `deploy/install_and_exec.py` | 내부 인덱스 설치 성공 후 executor 시작 |
| `deploy/Dockerfile` | Python·Codex·CSV·Skill을 이미지에 구성 |

9장은 Skill ZIP, 10장은 Plugin ZIP을 첨부했습니다. 여기서는 환경에 이미 있는 디렉터리를 capability_directories로 지정합니다. 다음 셀은 Skill 지침·진입 스크립트·설치/시작 코드를 읽을 뿐 실행하지 않습니다. 진입 스크립트는 사내 패키지의 calculate를 호출하고 세 결과 파일을 만듭니다. 코드·이미지·가상 패키지 인덱스는 기존 구성을 유지합니다.


In [ ]:
print(csv_text)
for name in ("deploy/skills/corporate-csv/SKILL.md",
             "deploy/skills/corporate-csv/scripts/run_report.py", "deploy/install_and_exec.py"):
    print(f"\n--- {name} ---")
    print((root / name).read_text(encoding="utf-8"))


### 키와 두 네트워크 경로

애플리케이션 키는 노트북에 두고 전용 환경 키만 임시 mount를 통해 executor의 CODEX_API_KEY로 전달합니다. 환경 키는 세션 소유자와 같은 조직·프로젝트·사용자 또는 서비스 계정에 속해야 합니다. 키 값·remote URL·전체 컨테이너 환경 변수를 출력하지 않습니다.

| 경로 | 역할 |
| --- | --- |
| executor → packages:8080 | 가상 내부망에서 wheel 설치 |
| executor → api.openai.com | 환경 등록을 위한 outbound HTTPS |
| executor → codex-cloud-environments.chatgpt.com | 명령/결과를 위한 outbound WebSocket |

실제 회사 VPN·인증·TLS와는 별도 실습입니다. Docker bridge는 연결 경로이며 사내 패키지 인덱스와 OpenAI 접속을 같은 의미로 해석하지 않습니다. [공식 인증·네트워크 조건](https://developers.openai.com/api/docs/guides/agents-api/environments/self-hosted#authentication)


In [ ]:
task = (
    "Read the corporate-csv Skill. Run its bundled script with "
    "/opt/venv/bin/python /opt/capabilities/skills/corporate-csv/scripts/run_report.py "
    "--input /workspace/sales.csv --output-dir /workspace/outputs. "
    "Use the preinstalled private acme-sales package; never rewrite or reinstall it. "
    "Inspect summary.json, report.md, and package_evidence.json. "
) + METRICS_CONTRACT


#### API 실행 허용 확인

설정 셀에서 `RUN_LIVE=True`를 선택한 뒤 이 준비 셀을 실행하세요. 기본 False에서는 여기서 오류 메시지와 함께 멈춥니다. 이후 API 셀에는 실행 여부 분기를 반복하지 않습니다. `requires_api` 태그는 오프라인 검증기용이며 Jupyter 자체의 접근 제어가 아닙니다.


In [ ]:
from openai import OpenAI

client = None
if not settings.enabled:
    raise RuntimeError(
        "Set RUN_LIVE=True to run the API example. Use the validation script for offline checks."
    )
if not docker_enabled:
    raise RuntimeError(("Self-hosted API requires RUN_DOCKER=True too;" " no API session created"))
key = executor_key(root)
if not key:
    raise RuntimeError(("dedicated_executor_key_required; no API " "session created"))
if key == settings.api_key:
    raise RuntimeError("executor_key_must_differ_from_application_key")
client = OpenAI(api_key=settings.api_key, timeout=90.0, max_retries=0)


### 이미지 준비와 입력 고정

API 준비 셀은 키와 실행 모드를 확인하며 원격 세션을 만들지 않습니다. Docker만 실습할 때는 그 셀을 건너뜁니다. 두 경로 모두 아래 이미지 준비를 세션 생성 전에 완료합니다. 최초 빌드는 공개 이미지/패키지를 다운로드할 수 있습니다.

CSV는 API 요청에 첨부하지 않고 이미지 빌드 때 복사합니다. 준비한 CSV 바이트와 빌드 입력을 비교한 뒤 해시를 기록합니다. CSV·Skill·패키지를 변경했다면 준비부터 이미지를 다시 빌드하고 새 실행을 시작하세요. 생성된 입력 해시가 실행 중인 컨테이너 바이트를 독립 증명하는 것은 아니며 이후 패키지 증거와 대조합니다.


In [ ]:
if docker_enabled:
    print("Preparing required worker, package-server and executor images...", flush=True)
    images = build_images(root, executor=True)
    build_input = (root / "deploy/.generated/data/sales.csv").read_bytes()
    if build_input != csv_bytes:
        raise RuntimeError("CSV changed during preparation; reload input and rebuild before creating a session")
    image_input_sha256 = hashlib.sha256(build_input).hexdigest()
    print("Required images ready:", sorted(images))
    print("Image build input SHA-256:", image_input_sha256)
else:
    print("Docker image preparation skipped. Set RUN_DOCKER=True to prepare images.")


## 2. 한눈에 보는 Self-hosted 사용 흐름

API가 반환한 환경 ID와 remote URL을 내가 운영하는 executor에 전달합니다. 아래 코드는 호출 형태를 보여주는 읽기용 발췌이며 별도 실행하지 마세요. 실제 요청과 제어 흐름은 다음 6단계에 직접 표시합니다.

```python
session = client.beta.agents.sessions.create(
    agent={"model": settings.model, "instructions": BASE_INSTRUCTIONS},
    environment={
        "type": "self_hosted",
        "workspace_directory": "/workspace",
        "capability_directories": ["/opt/capabilities"],
    },
)
# Pass session.environment.id and session.environment.remote_url to Docker.
```

순서: 세션 생성 → Docker 준비 → 이벤트 구독 후 executor 시작·연결 확인 → 입력 제출 → 결과 수집 → 정리. prepare_executor는 Docker 준비 helper이며 API 호출과 executor 시작을 하지 않습니다. 설치는 executor 시작 뒤 연결 전에 이루어집니다.


### 단계별 사용자 행동과 예상 결과

| 단계 | 사용자 행동 | 완료 확인 |
| --- | --- | --- |
| 1. 세션 생성 | ID 확인; 아직 작업을 보내지 않음 | Self-hosted 환경 ID |
| 2. Docker 준비 | 컨테이너 ID와 state 파일 경로 확인 | 아직 시작하지 않은 executor |
| 3. 연결 | 설치·연결 완료까지 대기 | 요청한 환경의 연결 확인 |
| 4. Skill 요청 | 입력을 한 번 보내고 완료까지 대기 | root Turn 실행·완료 상태 |
| 5. 결과 확보 | 로컬 파일 경로 확인 | 지표·보고서·패키지 증거 수집 |
| 6. 정리 | 마지막에 반드시 실행 | API 세션 부재·Docker 정리 |

정상 셀 사이에는 세션·컨테이너·임시 키 mount가 유지됩니다. 중간에 학습을 멈추더라도 6번 정리 셀을 실행하세요. 1–5단계는 오류·일반적인 중단 시 정리를 시도하지만 강제 커널 종료는 보장하지 않습니다. 오류 후 다음 단계로 진행하지 말고 정리 상태를 확인한 뒤 준비 셀부터 새 실행을 시작하세요. 생성·입력 셀을 반복 실행해 복구를 시도하지 마세요.

연결/작업 대기는 각 단계에서 시간을 새로 측정해 셀 사이 학습 시간을 제외합니다. 연결 스트림 timeout=30과 이벤트 기반 120초 예산, 작업 단계의 이벤트 기반 600초 예산은 엄격한 전체 실행 시간 제한이 아닙니다. 연결 timeout은 패키지 설치 중일 수도 있으므로 state 파일과 컨테이너 로그로 설치·연결 단계를 구분합니다.


### 이번 실행 상태 준비

이미지 준비 뒤 새 실행에서 한 번만 초기화합니다. 진행 중 이 셀을 다시 실행하면 자원 식별자를 잃을 수 있습니다. 반면 1단계는 상태를 초기화하지 않으며 생성 시도 기록으로 중복을 차단합니다. 준비/시작/입력 제출도 각각 한 번만 수행합니다. 중복 실행 오류 후에는 정리 상태를 확인하고 새 client·상태로 시작합니다.


In [ ]:
run_id = uuid4().hex
output_dir = settings.root / "outputs/deployment-api/12_self_hosted_api" / run_id
output_dir.mkdir(parents=True, exist_ok=True)
sessions, agents, attempts = {}, {}, []
cleanup_errors = []
resources = ExitStack()
session = provider = executor = result = turn = execution_context = None
connected = report_downloaded = False
input_submission_attempts = 0
executor_start_attempts = 0
downloads = None
metrics = evidence = items = None
docker_cleanup = "not_needed"


### 1. 세션 생성

환경 ID만 받고 아직 작업을 제출하지 않습니다. remote URL과 키는 출력하지 않습니다. 입력 해시가 이미지 준비 때와 같은지 확인하고 중복 생성은 차단합니다. 이 셀을 다시 실행해 기존 자원 식별자를 초기화하지 않습니다. 새 실행은 이전 실행을 정리한 뒤 준비부터 시작하세요.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors,
                      finalizers=(resources.close,)):
    if attempts or session is not None:
        raise RuntimeError("This run already attempted session creation. Do not resubmit.")
    if image_input_sha256 != hashlib.sha256(csv_bytes).hexdigest():
        raise RuntimeError("Prepare images for the fixed CSV bytes before creating a session")
    print("[1/6] Creating a self-hosted session without input...", flush=True)
    session = client.beta.agents.sessions.create(
        agent={"model": settings.model, "instructions": BASE_INSTRUCTIONS},
        environment={
            "type": "self_hosted",
            "workspace_directory": "/workspace",
            "capability_directories": ["/opt/capabilities"],
        },
        metadata=new_session_metadata(CHAPTER, run_id, attempts=attempts, output_dir=output_dir),
    )
    sessions[session.id] = {"deleted": False, "environment_id": session.environment.id}
    print("Session ID:", session.id)
    print("Environment ID:", session.environment.id)


### 2. Docker executor 준비

환경 ID와 remote URL을 Docker 준비 helper에 전달합니다. 컨테이너는 아직 시작하지 않습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors,
                      finalizers=(resources.close,)):
    if session is None or executor is not None:
        raise RuntimeError("Create a session first and prepare its executor only once")
    print("[2/6] Preparing Docker without starting the executor...", flush=True)
    provider, executor = resources.enter_context(prepare_executor(
        environment_id=session.environment.id,
        remote_url=session.environment.remote_url,
        key=key,
        root=root,
    ))
    print("Docker executor:", executor)
    print("Docker state file:", provider.output_dir / "state.json")
    print("Executor prepared; subscribe before starting it in step 3.")


### 3. 구독 후 시작·연결 확인

스트림을 먼저 구독한 뒤 준비한 executor를 한 번 시작합니다. 같은 Environment ID의 connected 이벤트를 확인해야 입력을 보낼 수 있습니다. ready는 hosted 환경의 연결 준비 이벤트로 연결 성공과 같게 취급하지 않습니다. 패키지 설치 실패 시 exec-server는 시작되지 않습니다.

연결 stream이 끝나거나 timeout이면 바로 입력을 재전송하지 말고 설치/연결 로그와 정리 상태를 확인합니다. 세션별 환경 ID와 executor가 필요하며 URL은 API가 반환한 그대로 사용합니다. [공식 Self-hosted 연결 흐름](https://developers.openai.com/api/docs/guides/agents-api/environments/self-hosted#start-the-executor)


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors,
                      finalizers=(resources.close,)):
    if executor is None or executor_start_attempts:
        raise RuntimeError("Prepare the executor first and start it only once")
    print("[3/6] Waiting for package installation and executor connection...", flush=True)
    started = time.monotonic()
    with client.beta.agents.sessions.events.stream(session.id, timeout=30) as connection_events:
        executor_start_attempts += 1
        provider.start(executor)
        for event in connection_events:
            if event.type == "agent.session.environment.connected":
                if event.environment.id != session.environment.id:
                    raise RuntimeError("Unexpected environment connection")
                connected = True
                print("Executor connected. Environment ID:", event.environment.id)
                break
            check_event(event)
            if time.monotonic() - started > 120:
                raise TimeoutError("Connection observation budget exceeded; no input was submitted")
    if not connected:
        raise RuntimeError("Executor did not connect; no input was submitted")
    execution_context = inspect_executor(
        provider, executor, expected_environment_id=session.environment.id
    )
    print("Input has not been submitted. Run step 4 to request the Skill.")


### 4. Skill 실행 요청

연결된 같은 세션에 작업을 한 번 제출합니다. 모델이 사내 패키지를 사용하는 번들 스크립트를 실행하도록 지시합니다. 연결 성공 자체는 작업 완료가 아닙니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors,
                      finalizers=(resources.close,)):
    if not connected or input_submission_attempts:
        raise RuntimeError("Connect first and submit input only once")
    print("[4/6] Submitting the Skill task to the connected environment...", flush=True)
    started = time.monotonic()
    input_submission_attempts += 1
    with client.beta.agents.sessions.stream(session.id, input=task).with_result_collection() as stream:
        for event in stream:
            if event.type == "agent.session.created":
                if event.session.id != session.id:
                    raise RuntimeError("Unexpected session in the task stream")
            elif event.type in {"agent.session.turn.in_progress", "agent.session.turn.completed"}:
                if event.turn.subagent_id is None:
                    print("Root turn:", event.turn.id, "Status:", event.turn.status, flush=True)
            elif event.type == "agent.session.turn.output_text.delta":
                pass
            elif event.type == "agent.session.requires_action":
                if any(a.type != "environment_connection" for a in event.session.required_actions):
                    raise RuntimeError("Unexpected required action")
            else:
                check_event(event)
            if time.monotonic() - started > 600:
                raise TimeoutError("Execution observation budget exceeded; do not resubmit input")
        result = stream.get_final_result()
    if result.session_id != session.id or result.turn_id != result.turn.id or result.turn.subagent_id is not None:
        raise RuntimeError("Result differs from the connected session or root Turn")
    sessions.setdefault(result.session_id, {"deleted": False})
    print("Returned turn ID:", result.turn_id)


### 5. 결과·명령 이력 확보

원래 세션/root Turn의 완료와 ID를 확인한 뒤 이력과 세 파일을 확보합니다. collect_report는 Docker의 파일 복사(docker cp)를 한 번씩 수행하는 helper이며 API artifact 다운로드 함수가 아닙니다. 이 예제의 수집 방식을 선택한 것이며 Self-hosted 전체의 파일 기능을 제한적으로 단정하지 않습니다.

| 결과 | 확보 경로 |
| --- | --- |
| 모델 최종 JSON | stream.get_final_result의 output_text |
| Turn/명령 이력 | Agents API turns/items 조회 |
| summary/report/package evidence | 이 실행의 executor에서 Docker 파일 복사 |

복사 도중 실패하면 완료로 표시하지 않습니다. 모델 응답과 복사한 summary는 다른 출처의 결과이며 정리 후 나란히 읽습니다. 여기서는 다시 분석을 실행하지 않습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors,
                      finalizers=(resources.close,)):
    report_downloaded = False
    if result is None or result.session_id != session.id:
        raise RuntimeError("Complete input submission in the connected session before collecting results")
    print("[5/6] Capturing the original turn, command history and files...", flush=True)
    turn = client.beta.agents.sessions.turns.retrieve(
        result.turn_id, session_id=result.session_id, timeout=15
    )
    if result.turn.status != "completed" or turn.status != "completed" or turn.id != result.turn_id:
        raise RuntimeError("Root turn completion was not confirmed")
    items = list(client.beta.agents.sessions.items.list(
        result.session_id, order="asc", limit=100, timeout=15
    ))
    downloads = output_dir / "downloads"
    metrics, evidence = provider.collect_report(executor, output_dir=downloads)
    report_downloaded = True
    print("Completed Turn:", turn.id, "| Status:", turn.status)
    print("Model response and copied summary are retained separately for local review.")
    print("Downloaded files:", downloads)
    print("Command records and container snapshot are retained for inspection after cleanup.")


### 6. 자원 정리·요약

중간에 학습을 멈춰도 실행하세요. API 세션을 먼저 정리하고 Docker와 임시 키 mount를 종료합니다. 원래 오류는 정리 실패로 덮지 않습니다.


In [ ]:
try:
    print("[6/6] Cleaning up the API session before Docker and the temporary key mount...", flush=True)
finally:
    original_error = sys.exc_info()[1]
    try:
        cleanup_errors[:] = cleanup_resources(client, sessions, agents, attempts)
    except Exception as cleanup_error:
        cleanup_errors.append(safe_error(cleanup_error))
    finally:
        try:
            resources.close()
            docker_cleanup = "verified" if provider is not None else "not_needed"
        except Exception as resource_error:
            cleanup_errors.append(safe_error(resource_error))
    if cleanup_errors:
        if original_error is None:
            raise RuntimeError("Resource cleanup incomplete; inspect local records")
        original_error.add_note("Resource cleanup incomplete; inspect local records")
if not all(owned.get("verified_absent") for owned in sessions.values()):
    raise RuntimeError("Session absence was not confirmed")
print({
    "executor_connected": connected,
    "input_submission_attempts": input_submission_attempts,
    "turn_status": turn.status if turn is not None else "not_completed",
    "report_downloaded": report_downloaded,
    "session_cleanup": "verified" if sessions else "not_needed",
    "docker_cleanup": docker_cleanup,
})
print("API session and Docker resources cleaned up. Inspect the retained evidence below.")


## 3. 보관한 실행 증거 확인

6단계가 끝나면 컨테이너는 삭제됐지만 items·컨테이너 식별 snapshot·다운로드 파일은 남습니다. 아래 확인 셀은 API와 Docker를 다시 호출하지 않습니다. 연결·실행 명령·결과 파일·정리를 구분해 읽으세요.


#### 실행 증거를 판단하는 기준

연결 성공, 명령 실행, 결과 파일을 순서대로 연결합니다. OpenAI의 Harness가 작업을 결정하고, 이 노트북에서 시작한 Docker의 exec-server가 명령을 실행합니다. 같은 숫자가 나온 최종 답변만으로 실행 위치를 판단하지 않습니다.

| 확인 대상 | 아래 실습에서 볼 정보 | 이 정보가 의미하는 것 |
| --- | --- | --- |
| 요청과 실행 환경 | Session ID → Environment ID → Docker container ID, 이름·이미지·hostname | 어떤 API 세션에 어느 컨테이너를 연결했는지 식별 |
| 연결 | `Executor connected`와 Docker에 전달된 `ENVIRONMENT_ID` | 해당 환경의 executor가 연결됨. 아직 작업 완료를 뜻하지 않음 |
| API 실행 이력 | `command_execution`의 `command`, `cwd`, `status`, `exit_code`, `output` | 모델의 설명과 구분되는 실제 명령·작업 디렉터리·실행 결과 |
| 컨테이너 결과 | `package_evidence.json`의 `module_file`, `input_sha256` | 설치된 사내 패키지의 경로와 사용한 입력을 Docker에서 복사한 파일로 확인 |

기대하는 실행 명령의 형태는 다음과 같습니다. 셸 래퍼나 따옴표가 추가될 수 있으므로 전체 명령을 읽어 확인합니다.

```bash
/opt/venv/bin/python \
  /opt/capabilities/skills/corporate-csv/scripts/run_report.py \
  --input /workspace/sales.csv \
  --output-dir /workspace/outputs
```

`cat run_report.py`는 파일을 읽은 기록입니다. Python으로 Skill 스크립트를 실행한 기록과 구분하세요. 완료된 명령의 종료 코드 0은 프로세스 성공을 의미하지만, 결과 내용까지 자동으로 보증하지는 않습니다. 입력 해시·패키지 정보·다운로드 파일도 함께 확인합니다. 명령 출력은 없거나 일부만 제공될 수 있으며, 없는 종료 코드는 0으로 해석하지 않습니다.

이 실습은 기존 items 조회와 Docker 결과 수집을 사용합니다. 별도의 Trace export나 실행 추적 helper를 추가하지 않습니다.


In [ ]:
if not report_downloaded or result is None or execution_context is None:
    raise RuntimeError("Complete result collection before reviewing retained evidence")
print("Model final response:", json.dumps(json.loads(result.output_text), indent=2))
print("Copied summary.json:", json.dumps(json.loads((downloads / "summary.json").read_text(encoding="utf-8")), indent=2))
print("Session:", result.session_id)
print("Turn:", result.turn_id)
print("Execution environment:", json.dumps(execution_context, indent=2))
print("Downloaded files:", downloads)

commands_for_turn = [
    item for item in items
    if item.type == "command_execution" and item.turn_id == result.turn_id
]
print("Captured command executions:", len(commands_for_turn))
if not commands_for_turn:
    print("No command execution items were captured; execution was not confirmed.")
for item in commands_for_turn:
    print("\nCommand item:", item.id)
    print("Command:", item.command)
    print("Working directory:", item.cwd)
    print("Status:", item.status)
    print("Exit code:", item.exit_code)
    print("Output:", item.output or "(No captured output)")

evidence = json.loads((downloads / "package_evidence.json").read_text(encoding="utf-8"))
print("\nPackage evidence:", json.dumps(evidence, indent=2))
print("Installed under /opt/venv:", evidence["module_file"].startswith("/opt/venv/"))
print(
    "Input SHA-256 matches:",
    evidence["input_sha256"] == hashlib.sha256(csv_bytes).hexdigest(),
)
for name in ("summary.json", "report.md", "package_evidence.json"):
    path = downloads / name
    print("Saved file:", path, "| Bytes:", path.stat().st_size)


#### 출력 해석과 확인 한계

- Python으로 번들 Skill을 실행한 명령인지 전체 내용을 읽습니다. 파일 읽기 명령만 있다면 Skill 실행을 확인한 것으로 표시하지 않습니다.
- 명령 상태가 completed이고 종료 코드가 0인지 확인합니다. 명령 이력이 없다는 메시지나 None 종료 코드는 실행 성공 증거가 아닙니다.
- 패키지 이름·버전과 설치 경로, 입력 SHA-256 일치 여부를 확인합니다. 해시가 다르면 입력 파일이나 결과의 실행 시점을 먼저 확인하세요. 같은 매출 숫자만으로 입력이 같다고 판단하지 않습니다.
- 아래 보고서 표시까지 정상 진행해 파일 내용도 확인합니다. 이 셀은 학습용 출력이며 정답 대조·보고서 완전성 검사·통과 기록을 대신하지 않습니다.

수동 실행은 다운로드 파일을 남기지만 검증 실행기의 summary.json 통과 기록을 자동으로 만들지는 않습니다. 커널을 재시작하면 메모리의 명령 이력은 사라집니다. 파일은 앞 셀에 표시된 이번 실행의 downloads 디렉터리에서 다시 읽을 수 있고, 상세 실행 검증 기록은 검증 스크립트를 실행한 경우에만 별도 결과 디렉터리에 남습니다. 이 코드가 표시하는 대상은 root Turn이며 하위 에이전트에 위임한 실행 이력 전체를 수집하는 예제는 아닙니다.


### 저장한 보고서 다시 읽기

다음 셀은 정리 후 로컬에 남은 보고서만 표시합니다. 앞의 명령 이력·패키지 증거와 연결해 읽으세요. API나 Docker를 다시 호출하지 않습니다.


In [ ]:
from IPython.display import Markdown, display

if not report_downloaded:
    raise RuntimeError("Complete all report copies before displaying the report")
display(Markdown((downloads / "report.md").read_text(encoding="utf-8")))


### 선택: 정리 전에 다른 터미널에서 관찰

2단계에서 받은 executor ID를 사용합니다. 3단계 이후에는 로그에서 설치·연결 진단을 읽을 수 있고 5단계 후 6단계 전에는 결과 파일을 직접 확인할 수 있습니다. 실행 중에 보려면 다음 읽기 명령만 다른 터미널에서 사용하세요.

```bash
docker ps --filter ancestor=agents-tutorial-executor:0.1.0 \
  --format 'table {{.ID}}\t{{.Names}}\t{{.Status}}'
docker logs --tail 50 <CONTAINER_ID>
docker exec <CONTAINER_ID> ls -l /workspace/outputs
docker exec <CONTAINER_ID> cat /workspace/outputs/package_evidence.json
```

이번 실행의 컨테이너 ID를 선택하세요. 작업 완료 전에는 파일이 없을 수 있고, Docker 로그가 모든 작업 명령을 담는 것은 아닙니다. Skill을 수동 실행해 결과를 섞지 마세요. API 명령 이력은 위 보관 증거로 확인합니다. 6단계 후 컨테이너가 보이지 않는 것은 정상이며 관찰을 위해 정리 코드를 제거하지 않습니다.


## 4. 선택 로컬 실습: 해시 거절·패키지 설치·Skill 실행

RUN_PACKAGE_CHECKS=True와 RUN_DOCKER=True일 때만 실행합니다. API가 요청한 작업이 아니라 별도의 worker에서 애플리케이션이 직접 실행하는 예행연습입니다. 기본 API의 완료 증거와 섞지 않습니다.

잘못된 해시 거절 → 올바른 해시 설치 → Skill 실행 → 결과 확인 → exec-server 도움말 확인 순서입니다. 공통 pip 인자는 인덱스·해시·의존성 정책을 직접 읽을 수 있게 유지합니다. 가상 사내 패키지는 acme-sales==0.1.0이며 실제 회사 패키지 서버는 아닙니다.


In [ ]:
if docker_enabled and RUN_PACKAGE_CHECKS:
    with DockerLab("12_self_hosted", root) as lab:
        print("[Local 1/4] Preparing the isolated package environment...", flush=True)
        network, worker = lab.package_environment()
        assert lab.assert_owned(network, network=True)["Internal"] is True
        pip_args = [
            "python", "-m", "pip", "install",
            "--index-url", "http://packages:8080/simple", "--trusted-host", "packages",
            "--no-deps", "--only-binary=:all:", "--require-hashes", "-r",
        ]
        print("[Local 2/4] Rejecting the wrong hash, then installing the verified wheel...", flush=True)
        bad = lab.execute(worker, pip_args + ["/opt/install/requirements.bad.txt"], check=False)
        assert bad.returncode != 0 and "HASHES" in bad.stderr.upper()
        lab.execute(worker, pip_args + ["/opt/install/requirements.txt"])
        no_app_key = "import os; assert not os.getenv('OPENAI_API_KEY'); assert not os.getenv('CODEX_API_KEY')"
        lab.execute(worker, ["python", "-c", no_app_key])
        print("[Local 3/4] Running the Skill directly, without an API request...", flush=True)
        lab.execute(worker, SKILL_COMMAND)
        local_metrics, local_evidence = lab.collect_report(worker)
        print("Local analysis result:", local_metrics)
        print("[Local 4/4] Checking exec-server command support...", flush=True)
        cli = lab.create("executor-help", EXECUTOR_IMAGE,
                         arguments=["--entrypoint", "codex"], command=["exec-server", "--help"])
        lab.start(cli)
        assert docker(["wait", cli]).stdout.strip() == "0"
        assert "exec-server" in docker(["logs", cli]).stdout
    print("Local check resources cleaned up.")
else:
    print("Optional local checks skipped. Set RUN_DOCKER=True and RUN_PACKAGE_CHECKS=True to run them.")


## 5. 문제 해결과 정리 확인

| 관찰한 상황 | 확인할 내용 |
| --- | --- |
| 이미지 준비가 오래 걸림 | 이미지/패키지 다운로드와 outputs/deployment-build 로그. 아직 API 세션 생성 전 |
| 3단계 연결 대기 | 패키지 설치·OpenAI 연결을 Docker 로그에서 구분. 설치 실패 시 exec-server는 시작되지 않음 |
| 컨테이너 mount 실패 | Docker Desktop이 볼 수 있는 프로젝트 공유 경로 사용. outputs/.executor-secrets는 임시 키 파일용 |
| 패키지가 import되지 않음 | 설치 entrypoint와 Skill이 모두 /opt/venv/bin/python을 사용하는지 확인 |
| 실행 성공 여부가 불명확 | 같은 입력을 즉시 다시 보내지 말고 Session·Turn·명령 이력·state 파일 확인 |
| 6단계 정리 실패 | API 취소/삭제와 Docker 종료 오류를 구분하고 이번 실행의 ID·label만 조사 |

CODEX_HOME은 컨테이너 내부 상태 경로입니다. Docker provider는 이번 실행의 label이 붙은 자원만 정리합니다. API 삭제 충돌은 bounded retry·취소 후 재확인으로 처리하며 원래 오류는 유지합니다.

컨테이너 state 파일은 자원 상태이지 Skill 실행 성공 증거가 아닙니다. 과거 날짜별 검증 이력은 학습 본문에서 분리했으며 새 실행은 위 단계별 출력으로 판단합니다. 강제 커널 종료에서는 finally가 실행되지 않을 수 있습니다.


## 완료 기준과 다음 장

세션과 환경·executor의 관계, 연결과 작업 완료의 차이, 모델 응답과 Docker 복사 파일, 명령·입력 바이트·패키지 증거를 구분할 수 있으면 완료입니다. 정리 후 로컬 파일과 보관한 snapshot으로 확인하며 API/Docker를 다시 호출하지 않습니다. 본문 수집 자료는 독립적인 서명이나 전체 감사 증명을 대신하지 않습니다.

[13장 관측](13_hooks_telemetry.ipynb)에서는 실행 상태와 애플리케이션 관측을 연결합니다. [14장 관찰자 재연결](14_operations.ipynb)은 OpenAI-hosted 관찰 복구이며 이 장의 executor 재시작·컨테이너 파일 복원을 구현하는 예제가 아닙니다. 선택 로컬 실습은 패키지/Skill 점검이지 API 실연결 검증이 아닙니다.

[실습 안내와 검증 범위](../../docs/validation.md)에서 실행 방법과 검사 범위를 확인하세요.


## 공식 참고 자료

### Agents API 전용 문서

- [Self-hosted sandboxes](https://developers.openai.com/api/docs/guides/agents-api/environments/self-hosted) — executor 설치·전용 키·네트워크·연결 이벤트.
- [Sandbox lifecycle](https://developers.openai.com/api/docs/guides/agents-api/environments/lifecycle) — compute 시작·재연결·종료와 API 세션 정리의 구분.
- [Sandbox security](https://developers.openai.com/api/docs/guides/agents-api/environments/security) — 격리·네트워크·실행 권한 검토·인증 정보 분리.
- [Events and items](https://developers.openai.com/api/docs/guides/agents-api/sessions/events) — 이벤트 스트림·완료 상태·저장된 items와 turns 조회.

### 보조 자료 — 공통 문서

- [Skills — Agents API 절](https://developers.openai.com/api/docs/guides/tools-skills#agents-api) — 공통 Skills 문서 중 사내 sandbox의 Skill 디렉터리와 capability_directories 등록에 해당하는 절입니다.

링크 확인: 2026-11-03. 문서 링크 확인은 API 실습 실행 검증과 별개입니다.
